# v3.1 표 청크 규칙 적용 기록 (2026-11-05)

## 배경
온톨로지 스키마 **v3.1이 확정**되었다. v3.1 §5-2(C8)는 표를 **행 단위로 청크**하고, 각 행 청크의 `text`에 **표 제목과 열 머리글을 반복**하며, ID를 `…:p{쪽4}:t{표2}r{행2}`로 쓰도록 정한다(§6).

## 변경 전 → 후
| 대상 | 변경 전 | 변경 후 |
|---|---|---|
| 홍역 신고기준 89쪽 (표 '항목 | 원문 내용') | `p0089:b01~b04` (문단 형식) | `p0089:t01r01~r04` (신고범위·진단기준·신고시기·신고방법) |
| 성홍열 신고기준 104쪽 | `p0104:b01~b04` | `p0104:t01r01~r04` |
| 홍역 지침 23쪽 (표 '구분 | 내용') | `p0023:b01~b02` | `p0023:t01r01`(신고범위), `t01r02`(신고시기) |
| 텍스트 | 항목 이름과 내용만 있음 | **표 제목 + 열 머리글 + 행 내용**을 같이 적음 |

## 변경하지 않은 것
- 일반 문단 블록(`b` 형식): 법 제2조·제11조·제12조·제13조, 홍역 93쪽, 24쪽 등은 문단이므로 그대로 둔다.
- **홍역 30쪽(집단시설별 관리)**: 병합 셀이 여러 시설 행에 걸쳐 있어 행별 귀속 기준을 정한 뒤 처리한다. 지금은 `b` 형식 그대로 두고 **보류**한다. (팀 결정 필요)

## 변경한 코드
- `src/preprocess/prevention_legal/pages.py`: `make_table_citation(key, page, excerpt, table, row)` 추가. 표 행의 `t{표}r{행}` 인용 ID를 만든다. 기존 `make_citation`은 문단용으로 유지.
- 노트북 `02_블록만들기.ipynb`: 신고기준 블록을 `t01r{번호}` ID와 표 제목·머리글로 생성하도록 수정.
- 노트북 `09_홍역23_30쪽_블록.ipynb`: 23쪽 블록을 `t01r{번호}` ID로 생성하도록 수정 (텍스트 머리글은 블록 파일에서 적용).
- 블록 파일 `blocks.jsonl`: 위 10개 블록만 바꿈. 나머지 블록은 그대로.

## 영향 확인
- `run.py`의 카드는 모두 문단 블록(`93쪽 b01`)을 가리키므로 변경 없음.
- 표 행을 가리키는 카드가 생기면 `expected_chunk_id`를 `t` 형식도 처리하도록 바꿔야 한다. 현재는 해당 카드가 없다.
- 팀 목록 항목 5(표 청크 규칙)는 **v3.1 확정으로 해소**된다. 팀에 확정 사실을 공유한다.

In [1]:
import json
from preprocess.prevention_legal.pages import make_table_citation

BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
recs = [json.loads(l) for l in open(BLOCKS, encoding="utf-8") if l.strip()]
table_ids = sorted(r["chunk_id"] for r in recs if ":t01r" in r["chunk_id"])
print("표 행 블록:", len(table_ids))
for cid in table_ids:
    print(" ", cid)
ids = [r["chunk_id"] for r in recs]
print("중복 ID:", len(ids) - len(set(ids)))

표 행 블록: 10
  kdca_diagnosis_reporting_2026:p0089:t01r01
  kdca_diagnosis_reporting_2026:p0089:t01r02
  kdca_diagnosis_reporting_2026:p0089:t01r03
  kdca_diagnosis_reporting_2026:p0089:t01r04
  kdca_diagnosis_reporting_2026:p0104:t01r01
  kdca_diagnosis_reporting_2026:p0104:t01r02
  kdca_diagnosis_reporting_2026:p0104:t01r03
  kdca_diagnosis_reporting_2026:p0104:t01r04
  kdca_measles_2026:p0023:t01r01
  kdca_measles_2026:p0023:t01r02
중복 ID: 0


In [2]:
# 표 행 인용 함수 확인: 89쪽 신고범위 행 (발췌문이 원문에 있어야 통과)
cit = make_table_citation("diagnosis", 89, "환자, 의사환자", table=1, row=1)
print(cit["chunk_id"], "| 쪽", cit["pdf_page"])

kdca_diagnosis_reporting_2026:p0089:t01r01 | 쪽 89
